# InventarisKu — Aplikasi Inventaris Barang Sekolah

Dashboard inventaris dengan tampilan modern, data contoh, pencarian/filter, tambah dan hapus barang, serta ekspor CSV.

**Cara menjalankan:** buka notebook ini di Google Colab, lalu jalankan sel kode di bawah dengan tombol ▶. Data demo hanya tersimpan selama sesi aktif; unduh CSV untuk menyimpan salinan data.

In [ ]:

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# ===== Data contoh inventaris sekolah =====
data_awal = [
    {"Kode":"INV-001","Nama Barang":"Laptop","Kategori":"Elektronik","Lokasi":"Lab Komputer","Jumlah":15,"Kondisi":"Baik","Tanggal Beli":"2025-01-15"},
    {"Kode":"INV-002","Nama Barang":"Proyektor","Kategori":"Elektronik","Lokasi":"Ruang Kelas 9A","Jumlah":2,"Kondisi":"Baik","Tanggal Beli":"2024-08-20"},
    {"Kode":"INV-003","Nama Barang":"Meja Siswa","Kategori":"Furnitur","Lokasi":"Ruang Kelas 8B","Jumlah":30,"Kondisi":"Baik","Tanggal Beli":"2023-07-10"},
    {"Kode":"INV-004","Nama Barang":"Kursi Siswa","Kategori":"Furnitur","Lokasi":"Ruang Kelas 8B","Jumlah":28,"Kondisi":"Perlu Perbaikan","Tanggal Beli":"2022-06-12"},
    {"Kode":"INV-005","Nama Barang":"Printer","Kategori":"Elektronik","Lokasi":"Ruang Tata Usaha","Jumlah":1,"Kondisi":"Rusak","Tanggal Beli":"2021-11-05"},
    {"Kode":"INV-006","Nama Barang":"Bola Basket","Kategori":"Olahraga","Lokasi":"Gudang Olahraga","Jumlah":8,"Kondisi":"Baik","Tanggal Beli":"2025-02-01"},
]
df = pd.DataFrame(data_awal)

# ===== Tampilan =====
display(HTML("""
<div style="background:linear-gradient(120deg,#123c8c,#1976d2);padding:24px;border-radius:18px;color:white;font-family:Arial;margin-bottom:14px">
  <div style="font-size:13px;letter-spacing:2px;opacity:.85">SISTEM INFORMASI SEKOLAH</div>
  <div style="font-size:30px;font-weight:800;margin:7px 0">📦 InventarisKu</div>
  <div style="font-size:15px;opacity:.92">Dashboard pengelolaan aset dan barang sekolah</div>
</div>
<div style="padding:12px 16px;border-left:5px solid #f4c542;background:#f4f8ff;border-radius:8px;color:#20345c;font-family:Arial">
<b>Petunjuk:</b> Tambahkan barang melalui formulir, cari data melalui kolom pencarian, pilih baris untuk menghapus, dan unduh laporan dalam format CSV.
</div>
"""))

# Dashboard cards
cards_out = widgets.Output()
def show_cards():
    with cards_out:
        clear_output(wait=True)
        total_jenis = len(df)
        total_unit = int(df["Jumlah"].sum()) if len(df) else 0
        baik = int(df.loc[df["Kondisi"]=="Baik","Jumlah"].sum()) if len(df) else 0
        perlu = int(df.loc[df["Kondisi"]=="Perlu Perbaikan","Jumlah"].sum()) if len(df) else 0
        rusak = int(df.loc[df["Kondisi"]=="Rusak","Jumlah"].sum()) if len(df) else 0
        display(HTML(f"""
        <div style="display:flex;flex-wrap:wrap;gap:10px;font-family:Arial;margin:12px 0">
          <div style="flex:1;min-width:130px;background:#eef5ff;border:1px solid #d8e6ff;border-radius:13px;padding:16px"><div style="color:#536782;font-size:12px">JENIS BARANG</div><div style="font-size:27px;font-weight:800;color:#174ea6">{total_jenis}</div></div>
          <div style="flex:1;min-width:130px;background:#effbf5;border:1px solid #cceedd;border-radius:13px;padding:16px"><div style="color:#536782;font-size:12px">TOTAL UNIT</div><div style="font-size:27px;font-weight:800;color:#14804a">{total_unit}</div></div>
          <div style="flex:1;min-width:130px;background:#fff9e8;border:1px solid #f7e6ae;border-radius:13px;padding:16px"><div style="color:#536782;font-size:12px">PERLU PERBAIKAN</div><div style="font-size:27px;font-weight:800;color:#a36a00">{perlu}</div></div>
          <div style="flex:1;min-width:130px;background:#fff0f0;border:1px solid #f5d0d0;border-radius:13px;padding:16px"><div style="color:#536782;font-size:12px">RUSAK</div><div style="font-size:27px;font-weight:800;color:#b42318">{rusak}</div></div>
        </div>
        """))
display(cards_out)

search = widgets.Text(placeholder="Cari nama, kode, kategori, atau lokasi...", description="🔎 Cari:", layout=widgets.Layout(width="70%"))
condition_filter = widgets.Dropdown(options=["Semua kondisi","Baik","Perlu Perbaikan","Rusak"], value="Semua kondisi", description="Kondisi:")
table_out = widgets.Output()

def render_table(*args):
    with table_out:
        clear_output(wait=True)
        view = df.copy()
        q = search.value.strip().lower()
        if q:
            mask = view.astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
            view = view[mask]
        if condition_filter.value != "Semua kondisi":
            view = view[view["Kondisi"] == condition_filter.value]
        if view.empty:
            display(HTML("<div style='padding:18px;color:#64748b'>Tidak ada data yang cocok dengan pencarian.</div>"))
        else:
            display(HTML(view.to_html(index=False, escape=True, border=0, classes="inventory-table")))
            display(HTML("""<style>.inventory-table{width:100%;border-collapse:collapse;font-family:Arial;font-size:13px}.inventory-table th{background:#164a9b;color:white;text-align:left;padding:11px}.inventory-table td{padding:10px;border-bottom:1px solid #e5edf7}.inventory-table tr:nth-child(even){background:#f4f8ff}</style>"""))

search.observe(render_table, names="value")
condition_filter.observe(render_table, names="value")
display(widgets.HBox([search, condition_filter]))
display(table_out)

display(HTML("<h3 style='font-family:Arial;color:#173b78;margin-top:22px'>➕ Tambah Barang Baru</h3>"))
kode = widgets.Text(description="Kode:", placeholder="INV-007", layout=widgets.Layout(width="48%"))
nama = widgets.Text(description="Nama:", placeholder="Contoh: Papan Tulis", layout=widgets.Layout(width="48%"))
kategori = widgets.Dropdown(options=["Elektronik","Furnitur","Alat Tulis","Olahraga","Laboratorium","Lainnya"], description="Kategori:", layout=widgets.Layout(width="48%"))
lokasi = widgets.Text(description="Lokasi:", placeholder="Contoh: Ruang Kelas 7A", layout=widgets.Layout(width="48%"))
jumlah = widgets.BoundedIntText(value=1, min=1, max=100000, description="Jumlah:", layout=widgets.Layout(width="48%"))
kondisi = widgets.Dropdown(options=["Baik","Perlu Perbaikan","Rusak"], description="Kondisi:", layout=widgets.Layout(width="48%"))
tanggal = widgets.DatePicker(description="Tgl. beli:", layout=widgets.Layout(width="48%"))
msg = widgets.Output()

def tambah_barang(_):
    with msg:
        clear_output(wait=True)
        if not kode.value.strip() or not nama.value.strip() or not lokasi.value.strip():
            display(HTML("<span style='color:#b42318'>Isi kode, nama barang, dan lokasi terlebih dahulu.</span>"))
            return
        if (df["Kode"].astype(str).str.upper() == kode.value.strip().upper()).any():
            display(HTML("<span style='color:#b42318'>Kode barang sudah digunakan. Gunakan kode yang berbeda.</span>"))
            return
        new_row = {"Kode":kode.value.strip().upper(),"Nama Barang":nama.value.strip(),"Kategori":kategori.value,
                   "Lokasi":lokasi.value.strip(),"Jumlah":jumlah.value,"Kondisi":kondisi.value,
                   "Tanggal Beli":str(tanggal.value) if tanggal.value else "-"}
        df.loc[len(df)] = new_row
        show_cards()
        render_table()
        display(HTML("<span style='color:#14804a;font-weight:700'>Barang berhasil ditambahkan ke tabel sesi ini.</span>"))
        kode.value = ""; nama.value = ""; lokasi.value = ""; jumlah.value = 1; tanggal.value = None

add_btn = widgets.Button(description="Simpan Barang", button_style="primary", icon="plus", layout=widgets.Layout(width="180px", height="40px"))
add_btn.on_click(tambah_barang)
display(widgets.VBox([widgets.HBox([kode,nama]), widgets.HBox([kategori,lokasi]), widgets.HBox([jumlah,kondisi]), tanggal, add_btn, msg]))

display(HTML("<h3 style='font-family:Arial;color:#173b78;margin-top:22px'>🗑️ Hapus Barang</h3><p style='font-family:Arial;color:#64748b;font-size:13px'>Masukkan kode barang yang ingin dihapus.</p>"))
hapus_kode = widgets.Text(placeholder="Contoh: INV-006", description="Kode:")
hapus_btn = widgets.Button(description="Hapus Barang", button_style="danger", icon="trash")
hapus_out = widgets.Output()
def hapus_barang(_):
    with hapus_out:
        clear_output(wait=True)
        global df
        target = hapus_kode.value.strip().upper()
        if not target:
            print("Masukkan kode barang.")
        elif not (df["Kode"].astype(str).str.upper() == target).any():
            print("Kode barang tidak ditemukan.")
        else:
            df.drop(df[df["Kode"].astype(str).str.upper() == target].index, inplace=True)
            df.reset_index(drop=True, inplace=True)
            show_cards()
            render_table()
            print(f"Barang dengan kode {target} berhasil dihapus dari sesi ini.")
hapus_btn.on_click(hapus_barang)
display(widgets.HBox([hapus_kode, hapus_btn]), hapus_out)

display(HTML("<h3 style='font-family:Arial;color:#173b78;margin-top:22px'>📥 Unduh Laporan</h3>"))
export_btn = widgets.Button(description="Unduh Inventaris (CSV)", button_style="success", icon="download")
export_out = widgets.Output()
def export_csv(_):
    with export_out:
        clear_output(wait=True)
        path = "/content/laporan_inventaris_sekolah.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        files.download(path)
export_btn.on_click(export_csv)
display(export_btn, export_out)

show_cards()
render_table()
display(HTML("""
<div style="margin-top:24px;padding:13px;border-radius:10px;background:#f4f8ff;color:#526581;font-family:Arial;font-size:12px">
<b>Catatan:</b> Versi demo ini menyimpan perubahan selama sesi Colab aktif. Unduh CSV secara berkala agar perubahan tersimpan. Data contoh dapat diganti dengan data inventaris sekolah sebenarnya.
</div>
"""))
